# Your First Generative Program: Structured Sentiment Classifier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibm-granite-community/mellea-cookbook/blob/main/recipes/SentimentClassifier/SentimentClassifier.ipynb)

This is the simplest useful generative program you can write with Mellea.

We will build a **customer review sentiment classifier** for laptop reviews that returns a structured result: label (`positive`, `negative`, or `neutral`) plus a confidence score. Along the way you will see exactly what Mellea adds over a plain LLM call, and why that matters even for a short program.

**What you will learn:**

- Why raw LLM responses are unreliable for structured output
- How `instruct(format=...)` gives you a validated, typed Pydantic object
- How to classify a batch of reviews in a loop

**Prerequisites:**

- Python 3.11 or 3.12
- [Ollama](https://ollama.com/) installed locally with `granite4.1:3b` pulled

Pull the model before running:

```bash
ollama pull granite4.1:3b
```

## Step 1. Set up your environment

You can run this notebook in [Colab](https://colab.research.google.com/), or download it and run locally.

To avoid Python package dependency conflicts, we recommend setting up a [virtual environment](https://docs.python.org/3/library/venv.html).

## Step 2. Set up a backend

### Option A - Local Ollama (no credentials required)

Run this cell to install Ollama inside Colab and pull the IBM Granite model. Skip to *Option B* if you prefer watsonx.ai.

In [ ]:
# Install Ollama
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Start the Ollama daemon in the background
import subprocess
import time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# Pull the IBM Granite model
!ollama pull granite4.1:3b

### Option B - IBM watsonx.ai

See [Getting Started with IBM watsonx](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_WatsonX.ipynb) for setup instructions.

You will need `WATSONX_URL`, `WATSONX_APIKEY`, and `WATSONX_PROJECT_ID`.

In [ ]:
# Uncomment if using watsonx.ai
# from ibm_granite_community.notebook_utils import get_env_var
# get_env_var("WATSONX_APIKEY")
# get_env_var("WATSONX_PROJECT_ID")
# get_env_var("WATSONX_URL")

## Step 3. Install relevant libraries

We will need `mellea`, `litellm` (for both the raw call and the Mellea backend), and `pydantic`.

In [ ]:
%pip install -q git+https://github.com/ibm-granite-community/utils
%pip install -q "mellea[litellm]" litellm pydantic

In [ ]:
import json
import re
from typing import Literal

import litellm
from litellm import completion
from mellea import start_session, MelleaSession
from mellea.backends.types import ModelOption
from pydantic import BaseModel

OLLAMA_MODEL = "ollama/ibm/granite4:micro"

clear_review = "The keyboard feels premium and the battery lasts all day. Absolutely worth the price."
ambiguous_review = "It arrived on time I guess. Works fine most of the time. Not sure if I'd buy again."

## Step 4. Start a Mellea session

A `MelleaSession` is the single entry point to the model. We also set `OLLAMA_MODEL` once so both approaches use the same backend.

In [ ]:
# Default: Ollama + Granite locally.
m: MelleaSession = start_session(
    backend_name="litellm",
    model_id=OLLAMA_MODEL,
)

# Uncomment for watsonx.ai:
# m = start_session(
#     backend_name="litellm",
#     model_id="watsonx/ibm/granite-4-h-small",
# )

print("Session ready:", m)

## Step 5. Define the output schema

Before calling the model, define exactly what a valid response looks like using a Pydantic model. This becomes the contract that Mellea enforces on every call.

- `label` is constrained to exactly three values via `Literal` - the model cannot return anything else.
- `confidence` is a float between 0 and 1.
- `reason` is a one-sentence explanation.

In [ ]:
class SentimentResult(BaseModel):
    label: Literal["positive", "negative", "neutral"]
    confidence: float   # 0.0 - 1.0
    reason: str         # one-sentence explanation

## Step 7. Raw LLM approach

Start with the direct approach: call Ollama via LiteLLM, ask for JSON, and parse the response manually.

In [ ]:
def _extract_json(text: str) -> str:
    match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    return match.group(1) if match else text.strip()


def classify_sentiment_raw(review: str) -> dict | None:
    prompt = f"""Classify the sentiment of this customer review.

Return ONLY valid JSON with this structure:
{{
  "label": "positive" | "negative" | "neutral",
  "confidence": 0.0,
  "reason": "string"
}}

Review: {review}
"""
    try:
        response: litellm.ModelResponse = completion(  # type: ignore[assignment]
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"},
        )
        content: str = response.choices[0].message.content or ""
        response_text = _extract_json(content)
        return json.loads(response_text)
    except json.JSONDecodeError as error:
        print(f"JSON parsing failed: {error}")
        return None
    except Exception as error:
        print(f"Raw extraction failed: {error}")
        return None

In [ ]:
print("RAW LLM APPROACH")
raw_result = classify_sentiment_raw(clear_review)

if raw_result:
    print(f"\nResult type:  {type(raw_result)}")
    print(f"Label:        {raw_result.get('label')}")
    print(f"Confidence:   {raw_result.get('confidence')}")
    print(f"Reason:       {raw_result.get('reason')}")
    print()
    print("The result is an unvalidated dict:")
    print("  - 'label' could be 'Positive', 'POSITIVE', or 'very positive'")
    print("  - 'confidence' could be 0.95, '0.95', or '95%'")
    print("  - Any key could be missing - .get() returns None silently")
else:
    print("\nExtraction failed")

## Step 8. Mellea approach

Now create a Mellea session over the same Ollama model. Pass the Pydantic class as `format=` to `m.instruct()`. Mellea handles prompt construction, model execution, JSON parsing, and schema validation. The return value is a validated `SentimentResult` instance - never a raw string or untyped dict.

The `ModelOption` enum provides backend-agnostic inference options. Using `ModelOption` keys ensures the same options work across all backends (Ollama, OpenAI, watsonx, etc.), making your code portable.

In [ ]:
def classify_sentiment_mellea(review: str) -> SentimentResult:
    result = m.instruct(
        "Classify the sentiment of this customer review. "
        "Provide a label (positive, negative, or neutral), "
        "a confidence score between 0 and 1, "
        "and a one-sentence reason: {{review}}",
        user_variables={"review": review},
        format=SentimentResult,
        model_options={ModelOption.SEED: 42},
    )
    assert result.value is not None, "instruct() returned no value"
    return SentimentResult.model_validate_json(result.value)

In [ ]:
print("MELLEA APPROACH")
mellea_result = classify_sentiment_mellea(clear_review)

print(f"\nResult type:  {type(mellea_result)}")
print(f"Label:        {mellea_result.label}")
print(f"Confidence:   {mellea_result.confidence:.0%}")
print(f"Reason:       {mellea_result.reason}")
print()
# label is always exactly 'positive', 'negative', or 'neutral' - enforced by Literal
# confidence is always a float - arithmetic is safe
print(f"label type:      {type(mellea_result.label).__name__}")
print(f"confidence type: {type(mellea_result.confidence).__name__}")

## Step 9. Compare on ambiguous input

Run both approaches on the ambiguous review to see how each handles uncertainty.

In [ ]:
print("TESTING WITH AMBIGUOUS INPUT")
print(f"Review: {ambiguous_review}")

print("\nRaw LLM:")
raw_amb = classify_sentiment_raw(ambiguous_review)
if raw_amb:
    print(f"  Result type: {type(raw_amb)}")
    print(f"  Label:       {raw_amb.get('label')}   (could be any casing or phrasing)")
    print(f"  Confidence:  {raw_amb.get('confidence')}   (could be float, string, or percent)")
else:
    print("  Failed")

print("\nMellea:")
mellea_amb = classify_sentiment_mellea(ambiguous_review)
print(f"  Result type: {type(mellea_amb)}")
print(f"  Label:       {mellea_amb.label}   (guaranteed to be one of: positive/negative/neutral)")
print(f"  Confidence:  {mellea_amb.confidence:.0%}   (guaranteed float)")
print(f"  Reason:      {mellea_amb.reason}")

## Step 10. Classify a batch of reviews

Because the output is a typed Python object, results can be collected into a list and aggregated using standard Python - no extra parsing or guards needed.

In [ ]:
reviews = [
    "The keyboard feels premium and the battery lasts all day. Absolutely worth the price.",
    "Stopped working after two weeks. Customer support was unhelpful and slow to respond.",
    "Decent product. Does what it says, nothing more. Packaging could be better.",
    "Incredible build quality. Fastest laptop I have ever owned.",
    "Screen flickering issue from day one. Very disappointed.",
]

results: list[SentimentResult] = [classify_sentiment_mellea(r) for r in reviews]

print(f"{'#':<3} {'Label':<10} {'Conf':>6}  Review")
print("-" * 72)
for i, (rev, res) in enumerate(zip(reviews, results), 1):
    print(f"{i:<3} {res.label:<10} {res.confidence:>5.0%}  {rev[:52]}...")

# Safe aggregation - confidence is always a float
avg_conf = sum(r.confidence for r in results) / len(results)
positives = sum(1 for r in results if r.label == "positive")
negatives = sum(1 for r in results if r.label == "negative")
print()
print(f"Positive: {positives}  Negative: {negatives}  Avg confidence: {avg_conf:.0%}")

## Output comparison

| | Raw LLM | Mellea |
|---|---|---|
| **Return type** | `dict` | `SentimentResult` (Pydantic model) |
| **Field access** | `result.get('label')` - no guarantee key exists | `result.label` - IDE autocomplete, type-checked |
| **Label values** | Any string the model produces | Exactly `positive`, `negative`, or `neutral` |
| **Confidence** | Could be float, string, or percent | Always a `float` - arithmetic is safe |
| **Parse failures** | Silent `None` return | `ValidationError` with field-level detail |
| **Downstream code** | Guards everywhere (`if result and 'label' in result`) | Trust the object - `result.label` always exists |

## Key takeaways

- Raw LLM calls return text that still needs parsing and validation
- Mellea returns typed objects that are easier to use in application code
- The `format=` parameter enforces the schema on every call with no extra code
- `ModelOption` keys ensure the same configuration works across all backends

The key shift: instead of parsing what the model returned, you declared what you need - and Mellea made the model conform to it.

**Next steps:**

- [Instruct, Validate, Repair](https://github.com/ibm-granite-community/mellea-cookbook/blob/main/recipes/InstructValidateRepair/InstructValidateRepair.ipynb): Add `Requirement` objects and automatic repair loops to enforce business rules beyond schema validation.
- [Structured Data Extraction](https://github.com/ibm-granite-community/mellea-cookbook/blob/main/recipes/StructuredDataExtraction/StructuredDataExtraction.ipynb): Apply the same patterns to invoices, emails, and support tickets with `@generative` stubs.